# 5. Findings

Every number here is reproducible from `data/sessions.parquet` and
`data/events.parquet` with the code in this repository. The claims are limited to
what a single user's logs can support, which is less than it looks like.

In [1]:
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.cwd().parent / "src"))
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

DATA = Path.cwd().parent / "data"
sessions = pd.read_parquet(DATA / "sessions.parquet")
events = pd.read_parquet(DATA / "events.parquet")
print(f"{len(sessions):,} sessions, {len(events):,} events")

1,213 sessions, 181,406 events


## 1. The prompt cache carries almost the entire input side

In [2]:
from agent_telemetry.analysis import economics

mix = economics.token_mix(sessions)
share = mix["cache_read_tokens"] / (
    mix["input_tokens"] + mix["cache_creation_tokens"] + mix["cache_read_tokens"]
)
print(f"cache reads are {share:.1%} of all input side tokens")
print(f"output tokens are {mix['output_tokens'] / mix.sum():.2%} of all tokens")

cache reads are 98.2% of all input side tokens
output tokens are 0.26% of all tokens


## 2. The cache pays off immediately, not eventually

The shortest fifth of sessions already sits above 90 percent. Restarting a
session to keep the context small discards the cache and can cost more than
continuing.

In [3]:
curve = economics.cache_efficiency_by_length(sessions)
curve[["sessions", "median_events", "cache_hit_rate"]]

,sessions,median_events,cache_hit_rate
0,224,7.0,0.9302
1,183,13.0,0.9311
2,201,25.0,0.9510
3,204,51.0,0.9519
4,198,113.5,0.9574
5,203,241.0,0.9875


## 3. Most sessions are short, and the mean describes none of them

In [4]:
from agent_telemetry.analysis import sessions as S

S.length_distribution(sessions)

,metric,mean,p50,p75,p90,p95,p99
0,events,149.6,33.0,113.0,223.0,349.2,2535.7
1,duration_minutes,41.6,2.4,3.4,9.7,39.1,616.0
2,tool_calls,37.6,10.0,41.0,79.8,110.4,406.3
3,total_tokens,14707996.3,1671516.0,8032397.0,16420763.8,25083016.8,366175926.2


## 4. Two tools are most of the work

In [5]:
from agent_telemetry.analysis import tools

top = tools.tool_frequency(events, top=5)
print(f"the top two tools are {top['share'].head(2).sum():.1%} of all calls")
top

the top two tools are 84.4% of all calls


,tool_name,calls,events,sessions,share
0,Bash,23343,23343,788,0.5122
1,Read,15126,15121,968,0.3319
2,Edit,3007,3007,99,0.0660
3,Write,1178,1178,90,0.0258
4,TaskUpdate,547,547,139,0.0120


## 5. Tool failures are rare and concentrated

In [6]:
tools.error_rates(events)

,results,errors,error_rate,median_session_error_rate,sessions_without_errors,sessions
0,45577,1186,0.026,0.0,753,1117


## 6. Most session files are subagents, and that is easy to misread

`isSidechain` is a property of the whole log file: a subagent gets its own file.
Counting flagged events as "sessions that used a subagent" conflates the agent
with its caller.

In [7]:
S.subagent_usage(sessions)

,metric,value
0,subagent sessions,1017.0000
1,share of all sessions,0.8384
2,events inside subagent sessions,67229.0000
3,share of all events,0.3706
4,"median events, subagent session",31.0000
5,"median events, main session",63.5000
6,"median tool calls, subagent session",10.0000
7,"median tool calls, main session",6.0000


In [8]:
S.delegating_sessions(sessions, events)

,metric,value
0,main sessions,196.000
1,main sessions that spawned an agent,29.000
2,share,0.148
3,median events when delegating,1286.000
4,median events otherwise,38.000


Delegating sessions are far larger than non-delegating ones, which is expected:
delegation happens in long sessions, because that is where the context pressure
that motivates it exists.

## What this dataset cannot tell you

- **Whether any model is better.** Models were chosen, not assigned. See notebook 4.
- **Whether an agent succeeded.** Nothing in the logs records the outcome of a task.
- **How long work took.** Duration measures elapsed wall clock time, including
  the hours a window sat open.
- **Anything generalizable.** This is one person, one machine, one working style,
  over two months. It describes that.

The pipeline, though, runs on anyone's logs. The interesting comparison is
between datasets, and that needs someone else to run it.